# Pothole model comparison on a Colab T4

Runs the YOLO models in a bundle (`pothole-colab*.zip`) on Colab's free T4 GPU, then checks
the result two ways:

1. **Same boxes?** Each T4 box is paired with the box the same model drew on the Mac.
2. **Same scores?** The T4's boxes are scored against the judged reference in the bundle.

The YOLO models do all the detecting. The reference labels (which boxes are real potholes and
cracks) came from blind reviewers and are frozen in `ai/evaluation/pooled/`, so this run checks
the detectors, not the labels. Each bundle's `colab_bundle.json` lists its models and clips.

**Before you start**

1. Put the zip where this notebook can find it: drag it into the **Files** panel (the folder
   icon on the left), or upload it to the top level of **My Drive**.
2. Check **Runtime > Change runtime type** shows **T4 GPU**.
3. Choose **Runtime > Run all**. Colab asks for Google Drive access only when the zip is not
   in the Files panel.

It takes about 5 to 10 minutes.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "This runtime has no GPU. Choose Runtime > Change runtime type > T4 GPU, "
    "then Runtime > Run all."
)
DEVICE = "cuda:0"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path


def newest(folder):
    return sorted(Path(folder).glob("pothole-colab*.zip"), key=lambda p: p.stat().st_mtime)


found = newest("/content")
if not found:
    from google.colab import drive

    drive.mount("/content/drive")
    found = newest("/content/drive/MyDrive")
assert found, (
    "No pothole-colab*.zip found. Drag it into the Files panel on the left (or upload it "
    "to the top level of My Drive), then run this cell again."
)
BUNDLE = found[-1]
WORK = Path("/content/pothole")
subprocess.run(["rm", "-rf", str(WORK)], check=True)
WORK.mkdir(parents=True)
subprocess.run(["unzip", "-q", str(BUNDLE), "-d", str(WORK)], check=True)
os.chdir(WORK)
MANIFEST = json.loads(Path("colab_bundle.json").read_text())
MODELS, CLIPS = MANIFEST["models"], MANIFEST["clips"]
print(f"{BUNDLE.name}: {MANIFEST['title']}")
print(f"{len(MODELS)} models; clips: {', '.join(CLIPS)}")

In [ ]:
# The version the Mac run used, so only the hardware differs.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.120"], check=True)

## 1. Run every model on the T4

In [ ]:
for clip, paths in CLIPS.items():
    Path("colab-preds", clip).mkdir(parents=True, exist_ok=True)
    for model, weights in MODELS.items():
        command = [
            sys.executable,
            "-m",
            "ai.evaluation.run_detector",
            "--weights",
            weights,
            "--images",
            paths["frames"],
            "--out",
            f"colab-preds/{clip}/{model}.json",
            "--imgsz",
            "640",
            "--threshold",
            "0.05",
            "--device",
            DEVICE,
        ]
        run = subprocess.run(command, capture_output=True, text=True)
        if run.returncode:
            print(run.stderr[-3000:])
            raise RuntimeError(f"{model} failed on {clip}")
        print(f"{clip:14} {model:24} {run.stdout.strip()}")

## 2. Did the T4 draw the same boxes as the Mac?

Boxes are paired one to one between the two runs: same class, overlap (IoU) of at least 0.5.
Only boxes at 0.25 or above count, because that is the lowest confidence any result uses. Tiny
score differences are normal between GPUs.

In [ ]:
def iou(a, b):
    w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - w * h
    return w * h / union if union > 0 else 0.0


def load(path):
    return {p["image"]: p for p in json.loads(Path(path).read_text())}


def compare(mac, t4, keep=0.25):
    paired = only_mac = only_t4 = 0
    worst = 0.0
    for image in mac:
        a, b = mac[image]["detections"], t4[image]["detections"]
        pairs = sorted(
            (
                (iou(x["xyxy"], y["xyxy"]), i, j)
                for i, x in enumerate(a)
                for j, y in enumerate(b)
                if x["label"] == y["label"]
            ),
            reverse=True,
        )
        used_a, used_b = set(), set()
        for overlap, i, j in pairs:
            if overlap < 0.5:
                break
            if i in used_a or j in used_b:
                continue
            used_a.add(i)
            used_b.add(j)
            if max(a[i]["score"], b[j]["score"]) >= keep:
                paired += 1
                worst = max(worst, abs(a[i]["score"] - b[j]["score"]))
        only_mac += sum(x["score"] >= keep for i, x in enumerate(a) if i not in used_a)
        only_t4 += sum(y["score"] >= keep for j, y in enumerate(b) if j not in used_b)
    return paired, only_mac, only_t4, worst


def ms_per_frame(run):
    return sum(p["inference_ms"] for p in run.values()) / len(run)


print(
    f"{'clip':14} {'model':24} {'same box':>8} {'Mac only':>8} {'T4 only':>8}"
    f" {'max score diff':>14} {'ms/frame Mac':>12} {'T4':>5}"
)
totals = [0, 0, 0]
for clip, paths in CLIPS.items():
    for model in MODELS:
        mac = load(f"{paths['macPreds']}/{model}.json")
        t4 = load(f"colab-preds/{clip}/{model}.json")
        assert mac.keys() == t4.keys(), f"{clip}: the two runs saw different frames"
        paired, only_mac, only_t4, worst = compare(mac, t4)
        totals = [totals[0] + paired, totals[1] + only_mac, totals[2] + only_t4]
        print(
            f"{clip:14} {model:24} {paired:8d} {only_mac:8d} {only_t4:8d} {worst:14.4f}"
            f" {ms_per_frame(mac):12.1f} {ms_per_frame(t4):5.1f}"
        )
print()
print(f"{totals[0]} boxes on both, {totals[1]} only on the Mac, {totals[2]} only on the T4.")

## 3. Score the T4's boxes against the judged reference

Each T4 box is placed on the judged cluster it overlaps, by the same rule that built the
clusters. A box that lands on no judged cluster cannot be scored, so it is counted, not hidden.

In [ ]:
command = [
    sys.executable,
    "-m",
    "ai.evaluation.score_sets",
    "--config",
    MANIFEST["scoring"]["config"],
    "--preds-root",
    "colab-preds",
    "--json",
    "colab-scores.json",
]
score = subprocess.run(command, capture_output=True, text=True)
if score.returncode:
    print(score.stderr[-3000:])
    raise RuntimeError("scoring failed")
Path("colab-scores.txt").write_text(score.stdout)


def key(row):
    return row["target"], row["detector"], row["mode"], row["threshold"]


mac = {key(r): r for r in json.loads(Path(MANIFEST["scoring"]["macScores"]).read_text())}
t4 = {key(r): r for r in json.loads(Path("colab-scores.json").read_text())}
changed = [k for k in mac if {field: t4[k][field] for field in mac[k]} != mac[k]]

print("T4 run, all clips pooled, unsure counted as wrong:")
print(f"{'target':8} {'detector':52} {'t':>5} {'precision':>9} {'recall':>6} {'unjudged':>8}")
for target, detector, threshold in MANIFEST["summary"]:
    r = t4[(target, detector, "strict", threshold)]
    print(
        f"{target:8} {detector:52} {threshold:5} {r['precision']:9.2f} {r['recall']:6.2f}"
        f" {r['unjudged_boxes']:8d}"
    )
print()
if changed:
    print(f"{len(changed)} of {len(mac)} score rows differ from the Mac run, for example:")
    for k in changed[:5]:
        print(
            " ",
            k,
            "Mac",
            mac[k]["precision"],
            mac[k]["recall"],
            "T4",
            t4[k]["precision"],
            t4[k]["recall"],
        )
else:
    print(f"All {len(mac)} score rows are identical to the Mac run.")
print("Full tables: colab-scores.txt (open it from the Files panel on the left).")

## What this shows

If the boxes pair up and the scores are identical, the comparison in
`docs/model-comparison.md` does not depend on the machine it ran on: a T4 changes the speed,
not the answer. Better accuracy has to come from fine-tuning on labelled Tashkent footage
(`ai/training/`), which is what a T4 is worth its time for.